# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">9/29(화) 오전 · 정규표현식 · 탐지 룰 — 실습</mark>

오늘 오전의 도착점은 **쉼표가 없는 서버 로그를 읽어 `normalized_logs.json` 으로 남기고, 수상한 계정과 IP 를 찾아내는 것**입니다.

어제까지 쓰던 로그는 쉼표로 곱게 나뉘어 있었습니다. 오늘 도착한 로그는 그렇지 않습니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 내 사본 만들기

1. 위 메뉴에서 파일 › 드라이브에 사본 저장을 누릅니다.
2. 제목이 「사본: …」으로 바뀌면 된 것입니다.

### 0.2 오늘 오전의 순서

| 교시 | 무엇 |
|---|---|
| 2교시 | `split` 이 조용히 틀린다 · 정규식 기호 여섯 개 |
| 3교시 | `re.search` 로 찾기 · 이름 붙인 괄호로 네 값 꺼내기 |
| 4교시 | 안 맞는 줄 남기기 · 정규화 파일 · 탐지 룰 ① 브루트포스 · ② 의심 IP |

### 0.3 막혔을 때

1. 문제 아래 **💡 힌트**를 순서대로 따라 합니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다.
3. 도전 문제는 안 풀고 넘어가도 됩니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · 정규표현식 ①</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **비정형 로그** | 정형(CSV)과 견주면 무엇이 다른가 |
| **정규표현식** | 위치가 아니라 생김새로 찾는다는 것이 무슨 뜻인가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 비정형 로그 →
- 정규표현식 →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1 · 값의 번호로 찾으면 조용히 틀린다</mark>


### 왜 필요한가

1. 어제까지 로그는 `09:01,kim01,LOGIN_OK,10.0.3.21` 처럼 쉼표로 나뉘어 있었습니다. 누가 정리해 준 파일이었습니다.
2. 서버가 직접 남기는 기록은 쉼표가 없습니다. 한 줄이 통째로 옵니다.
3. 빈칸으로 잘라 「몇 번째 칸」으로 세면, 줄마다 모양이 조금만 달라져도 **에러 없이 틀린 값**이 들어옵니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 정규표현식 | 찾을 글자의 **모양**을 규칙으로 적는 표기 |
| 메타문자 | 글자 자체가 아니라 뜻을 갖는 기호 (`\d` `+` `.`) |
| `re` | 파이썬에서 정규표현식을 다루는 기본 도구 |
| `re.search` | 문자열에서 규칙에 맞는 **첫 부분**을 찾는다 |


### 0.4 먼저 · 드라이브를 연결하고 `agent_core` 로 들어갑니다

아래 두 셀을 **맨 먼저** 실행합니다. 계정 선택 창이 뜨면 코랩을 연 계정을 고르고 허용을 누릅니다.
이 자리를 잡아 두면 오늘 만드는 파일이 **전부 드라이브에 바로 저장**됩니다. 나중에 옮길 일이 없습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!mkdir -p /content/drive/MyDrive/agent_core
%cd /content/drive/MyDrive/agent_core
!ls


아래 셀을 먼저 실행합니다. 오늘 오전 내내 이 파일을 씁니다.


In [1]:
%%writefile raw_logs.txt
2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11
2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5
2026-09-29 09:15:31 INFO accepted login for lee.yh from 10.1.2.34
2026-09-29 09:16:02 INFO session closed for 10.1.2.34
2026-09-29 10:03:19 WARN failed login for park.js from 10.1.3.7
2026-09-29 10:03:31 INFO accepted login for park.js from 10.1.3.7
2026-09-29 11:20:55 INFO accepted login for choi.mk from 10.1.4.2
2026-09-29 12:40:12 INFO session closed for 10.1.4.2
2026-09-29 03:11:05 WARN failed login for admin from 211.45.12.9
2026-09-29 03:12:47 WARN failed login for admin from 211.45.12.9
2026-09-29 03:13:58 WARN failed login for admin from 211.45.12.9
2026-09-29 03:15:22 WARN failed login for admin from 211.45.12.9
2026-09-29 03:17:09 INFO accepted login for admin from 211.45.12.9
2026-09-29 14:05:38 INFO accepted login for jung.hw from 10.1.2.88
2026-09-29 22:14:03 WARN failed login for kim.cs from 185.220.101.34
2026-09-29 22:14:21 WARN failed login for lee.yh from 185.220.101.34
2026-09-29 22:14:40 WARN failed login for choi.mk from 185.220.101.34
2026-09-29 22:14:58 WARN failed login for jung.hw from 185.220.101.34
2026-09-29 22:15:12 INFO session closed for 185.220.101.34
2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34


Writing raw_logs.txt


In [ ]:
!cat raw_logs.txt


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 `split` 이 조용히 틀린다</mark>

빈칸으로 자르면 리스트가 됩니다. 칸 번호로 값을 꺼낼 수 있습니다.

```python
line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
parts = line.split()
print(parts[6])      # kim01  ← 6번 칸이 계정이다
```

문제는 **줄마다 칸 개수가 다를 때**입니다. 번호가 밀려도 파이썬은 아무 말도 하지 않습니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
line = "2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11"
parts = line.split()
print(parts[6])
```

막히면 바로 위 `2.1 split 이 조용히 틀린다` 설명을 다시 봅니다.


In [2]:
line = "2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11"
parts = line.split()
print(parts[6])


kim.cs


✅ `kim.cs`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-2 · 무엇이 보일까요</font></h3></td></tr></table>

같은 번호를 **다른 줄**에 써 봅니다. 이 줄에는 `invalid user` 라는 말이 끼어 있습니다.

```python
line = "2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34"
parts = line.split()
print(parts[6])
```

막히면 바로 위 `2.1 split 이 조용히 틀린다` 설명을 다시 봅니다.


In [3]:
line = "2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34"
parts = line.split()
print(parts[6])


invalid


✅ `invalid`


계정 이름이 나와야 할 자리에서 `invalid` 가 나왔습니다. **에러가 나지 않습니다.** 틀린 값이 조용히 들어갑니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-3 · 칸 개수 세기</font></h3></td></tr></table>

두 줄을 빈칸으로 잘라 **칸이 몇 개인지** 각각 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `9` · `11` |

**💡 힌트**

1. 빈칸으로 자를 때는 `split()` 안에 아무것도 넣지 않습니다.
2. 리스트의 개수는 `len()` 으로 셉니다.
3. 주어진 **두 줄을 각각** 잘라 `print(len(parts))` 를 두 번 합니다.


In [4]:
line1 = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
line2 = "2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-4 · 맨 뒤 칸 꺼내기</font></h3></td></tr></table>

두 줄에서 **IP** 를 각각 출력하시오. IP 는 언제나 맨 뒤 칸입니다.

- 맨 뒤 칸의 번호는 `len()` 에서 1을 뺀 값입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `203.0.113.5` · `185.220.101.34` |

**💡 힌트**

1. 먼저 잘라서 리스트를 만듭니다.
2. 맨 뒤 번호는 `len(parts) - 1` 입니다.
3. 그 번호를 대괄호에 넣습니다.


In [9]:
line1 = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
line2 = "2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34"
print(len(line1.split()))
print(len(line2.split()))

9
11


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-5 · 파일 전체에서 6번 칸 보기</font></h3></td></tr></table>

`raw_logs.txt` 를 한 줄씩 읽어 **6번 칸**을 모두 출력하시오. 계정이 아닌 값이 섞여 나오는 것을 눈으로 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 계정 이름들 사이에 `10.1.2.34`·`invalid` 같은 값이 섞여 나온다 |

**💡 힌트**

1. 파일 읽기는 어제와 같습니다. `with open(...) as f:` 와 `for line in f:`
2. 반복 안에서 빈칸으로 자릅니다.
3. 6번 칸을 출력합니다.


In [10]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.2 기호로 글자 모양을 적는다</mark>

| 기호 | 뜻 | |
|---|---|---|
| `\d` | 숫자 한 글자 | 코드에 쓴다 |
| `\w` | 글자·숫자·밑줄 한 글자 | 코드에 쓴다 |
| `[\d.]` | 대괄호 안의 글자 중 아무거나 한 글자 (여기서는 숫자 또는 점) | 코드에 쓴다 |
| `+` | 왼쪽 기호를 하나 이상 이어 붙인다 | 코드에 쓴다 |
| `.` | 아무 글자나 한 글자 | 읽을 수만 있으면 된다 |
| `{1,3}` | 왼쪽 기호를 **1개 이상 3개 이하**로 이어 붙인다 | 오늘은 안 쓴다 |

- `+` 는 왼쪽 기호를 이어 붙이다가 **그 기호에 맞지 않는 글자에서 멈춥니다.** 공백이 기준이 아닙니다.
- 이 기호들을 실제로 쓰는 것은 **3교시**입니다. 여기서는 모양만 눈에 익혀 둡니다.


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-6 · 계정이 아닌 줄을 세기</font></h3></td></tr></table>

6번 칸에 **점(`.`)이 들어 있지 않은 값**이 몇 개인지 세어 출력하시오. 계정 이름은 점이 있는 것도 없는 것도 있어서 정확하지 않습니다. 그 부정확함을 직접 확인하는 문제입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 숫자 하나 |

**💡 힌트**

1. 숫자를 담을 이름을 반복 전에 만듭니다.
2. 글자가 들어 있는지는 `in` 으로 확인합니다.
3. `if "." not in parts[6]:` 처럼 씁니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-7 · WARN 인 줄만 세기</font></h3></td></tr></table>

`raw_logs.txt` 에서 등급이 **`WARN`** 인 줄이 몇 개인지 세어 출력하시오. 정규식을 쓰지 않고 `in` 으로도 됩니다. 두 방법 중 하나를 고르세요.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `WARN 11줄` |

**💡 힌트**

1. 숫자를 담을 이름을 반복 전에 만듭니다.
2. `if "WARN" in line:` 이 가장 쉽습니다.
3. f-string 으로 문장을 만들어 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `line.split()` | 빈칸으로 자른다. **값의 번호가 밀리면 조용히 틀린다** |
| `import re` | 정규표현식 도구를 부른다 |
| `\d` `\w` `[ ]` `+` | 이 교시에 눈에 익힌 기호 네 개 |
| `re.search` · `re.findall` | **3교시**에서 배운다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">3교시 (11:00–11:50) · 그룹으로 꺼낸다</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **그룹(group)** | 패턴에서 괄호로 묶으면 무엇이 달라지나 |
| **named group** | 괄호에 이름을 붙이면 무엇이 편해지나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 그룹(group) →
- named group →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2 · 한 줄을 딕셔너리 한 개로</mark>


### 왜 필요한가

1. 지금까지는 `re.search` 로 처음 맞는 덩어리 하나만 꺼냈습니다. 이제 한 줄에서 **네 가지를 한꺼번에** 꺼내야 합니다.
2. 규칙 안에서 꺼내고 싶은 자리를 괄호로 묶으면 그 부분만 따로 받을 수 있습니다. 이것을 **그룹**이라고 합니다.
3. 괄호가 넷이면 몇 번째가 무엇인지 또 외워야 합니다. 그래서 괄호에 **이름**을 붙입니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 그룹 | 규칙 안에서 따로 꺼내고 싶은 부분을 묶은 괄호 |
| `.group(1)` | 첫 번째 괄호가 잡은 부분 |
| 이름 붙인 그룹 | `(?P<user>...)` 처럼 괄호에 이름을 단 것 |
| `.groupdict()` | 이름과 값을 짝지은 **딕셔너리**를 돌려준다 |
| `None` | 못 찾았다는 뜻. 여기서 `.group()` 을 부르면 오류가 난다 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.1 `re.search` 와 `re.findall`</mark>

```python
import re

m = re.search(r"\d+", "확인 필요: admin 실패 3회")
print(m.group())     # 3
```

- 패턴 앞의 `r` 은 역슬래시를 파이썬이 먼저 해석하지 않게 하는 **raw string 표기**입니다. 정규식에는 붙여 쓰는 습관을 들입니다.
- `re.search` 는 **처음 맞는 것 하나**를 돌려줍니다. `.group()` 으로 꺼냅니다.
- `re.findall` 은 **맞는 것 전부**를 돌려줍니다. 결과가 하나여도 **리스트**로 옵니다. 못 찾으면 빈 리스트 `[]` 입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-1 · IP 를 찾기</font></h3></td></tr></table>

주어진 줄에서 **IP** 를 찾아 출력하시오. 번호를 쓰지 않고 글자 모양으로 찾습니다.

- 오늘 로그의 IPv4 주소는 숫자와 점으로 되어 있습니다. `[\d.]+` 는 주소를 검증하지 않고 그 모양의 글자를 찾습니다.
- 줄 맨 앞 날짜에도 숫자가 있으니, 앞의 `from` 까지 함께 적어 자리를 못박습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `from 203.0.113.5` |

**💡 힌트**

1. 숫자 또는 점 한 글자는 `[\d.]` 입니다.
2. 이어진 만큼 잡으려면 뒤에 `+` 를 붙입니다.
3. 패턴 앞에 `from ` 을 글자 그대로 적습니다.
4. 찾은 결과는 `.group()` 으로 꺼냅니다.


In [14]:
import re

line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
result = re.search(r"from [\d.]+", line)
print(result.group())

from 203.0.113.5


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-2 · 계정을 찾기</font></h3></td></tr></table>

같은 줄에서 **계정**을 찾아 출력하시오. 문제 3-1과 모양이 같습니다.

- 계정에는 `kim.cs` 처럼 점이 들어갈 수 있습니다.
- 계정은 `login for` 뒤에 있으니 앞의 `for` 까지 함께 적습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `for kim01` |

**💡 힌트**

1. 글자 또는 점 한 글자는 `[\w.]` 입니다.
2. 이어진 만큼 잡으려면 뒤에 `+` 를 붙입니다.
3. 패턴 앞에 `for ` 를 글자 그대로 적습니다.


In [15]:
# 여기에 코드를 입력하세요
result = re.search(r"for [\w.]+", line)
print(result.group())

for kim01


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-3 · 파일 전체에서 IP 찾기</font></h3></td></tr></table>

`raw_logs.txt` 를 한 줄씩 읽어 문제 3-1의 패턴을 **`re.findall`** 로 찾아 출력하시오.

- `re.search` 는 못 찾으면 `None` 을 주어 `.group()` 에서 오류가 납니다.
- `re.findall` 은 못 찾으면 **빈 리스트 `[]`** 를 주므로 오류가 나지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 17줄은 `['from 10.1.2.11']` 처럼, `from` 이 없는 3줄은 `[]` |

**💡 힌트**

1. 파일을 한 줄씩 읽는 뼈대는 문제 2-5와 같습니다.
2. 줄마다 `re.findall(r"from [\d.]+", line)` 을 부릅니다.
3. 돌려받은 리스트를 그대로 출력합니다.


In [17]:
# 여기에 코드를 입력하세요
import re

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        result = re.findall(r"from [\d.]+", line)
        print(result)

['from 10.1.2.11']
['from 203.0.113.5']
['from 10.1.2.34']
[]
['from 10.1.3.7']
['from 10.1.3.7']
['from 10.1.4.2']
[]
['from 211.45.12.9']
['from 211.45.12.9']
['from 211.45.12.9']
['from 211.45.12.9']
['from 211.45.12.9']
['from 10.1.2.88']
['from 185.220.101.34']
['from 185.220.101.34']
['from 185.220.101.34']
['from 185.220.101.34']
[]
['from 185.220.101.34']


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.2 괄호에 이름을 붙여 꺼내기</mark>

꺼내고 싶은 자리를 **소괄호로 묶고**, 괄호 바로 안에 `?P<이름>` 을 적습니다. 모양은 늘 같습니다.

```python
import re

line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

m = re.search(PATTERN, line)
print(m.group("user"))     # kim01
print(m.groupdict())       # {'time': '09:12:00', 'level': 'WARN', 'user': 'kim01', 'ip': '203.0.113.5'}
```

- 괄호 **안** 네 곳만 값으로 꺼내집니다. 괄호 **밖**의 `login for` 와 `from` 은 줄에 있어야 맞지만 값으로는 안 나옵니다.
- `.group("user")` 처럼 **이름으로** 꺼냅니다. 번호를 세지 않아도 읽으면 무엇인지 압니다.
- `.groupdict()` 를 부르면 **딕셔너리가 그대로** 나옵니다. 줄글 한 줄이 딕셔너리 한 개가 되는 자리입니다.
- ⚠ 이 패턴을 **처음부터 치지 마세요.** 위 `PATTERN` 을 복사해 쓰고, 이름 하나만 바꿔 보는 것으로 충분합니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-4 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
import re

line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

m = re.search(PATTERN, line)
print(m.group("ip"))
```

막히면 바로 위 `3.2 괄호에 이름을 붙여 꺼내기` 설명을 다시 봅니다.


In [18]:
import re

line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

m = re.search(PATTERN, line)
print(m.group("ip"))


203.0.113.5


✅ `203.0.113.5`

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-5 · 네 이름을 붙이기</font></h3></td></tr></table>

주어진 줄에서 **시각 · 등급 · 계정 · IP** 네 가지를 이름 붙인 괄호로 꺼내 `.groupdict()` 를 출력하시오.

- 이름은 **`time` · `level` · `user` · `ip`** 로 정해져 있습니다. 오늘 산출물이 이 이름을 씁니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{'time': '09:12:00', 'level': 'WARN', 'user': 'kim01', 'ip': '203.0.113.5'}` |

**💡 힌트**

1. 괄호 바로 안에 `?P<이름>` 을 적습니다.
2. 시각은 `[\d:]+`, 등급과 `failed`·`accepted` 자리는 `\w+`, 계정은 `[\w.]+`, IP 는 `[\d.]+` 입니다.
3. 마지막에 `.groupdict()` 를 출력합니다.
4. 막히면 **아래 패턴을 그대로 복사해** 쓰세요 — 오늘은 읽을 수 있으면 됩니다.

```python
PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"
```


In [21]:
# 여기에 코드를 입력하세요
import re

line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

m = re.search(PATTERN, line)
print(m.groupdict())

{'time': '09:12:00', 'level': 'WARN', 'user': 'kim01', 'ip': '203.0.113.5'}


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-6 · 파일 전체를 딕셔너리로</font></h3></td></tr></table>

`raw_logs.txt` 를 한 줄씩 읽어 맞는 줄만 딕셔너리로 바꿔 리스트에 모으고, 개수를 출력하시오.

- 패턴은 문제 3-5와 같습니다. 맨 위에 `PATTERN` 이라는 이름으로 한 번만 적어 두고 씁니다.
- 못 찾은 줄은 이번에는 그냥 넘어갑니다. 다음 교시 4.1에서 다룹니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `정규화 16건` |

**💡 힌트**

1. 빈 리스트를 반복 전에 만듭니다.
2. `if m:` 으로 찾았을 때만 담습니다.
3. `m.groupdict()` 를 그대로 `append` 합니다.


In [23]:
# 여기에 코드를 입력하세요
import re

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

records = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            records.append(m.groupdict())

print("정규화", len(records), "건")

정규화 16 건


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-7 · 계정만 모아 보기</font></h3></td></tr></table>

문제 3-6의 결과에서 **계정 이름만** 리스트로 모아 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 계정 이름 16개가 담긴 리스트 |

**💡 힌트**

1. 문제 3-6의 코드를 그대로 쓰고 뒤에 이어 붙입니다.
2. `rows` 를 돌면 딕셔너리가 하나씩 나옵니다.
3. `row["user"]` 를 새 리스트에 `append` 합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `re.search(r"패턴", 글자).group()` | 맞는 첫 부분을 찾아 글자를 꺼낸다 |
| `re.findall(r"패턴", 글자)` | 맞는 것을 모두 리스트로 준다. 없으면 `[]` |
| `(?P<user>...)` | 꺼낼 자리를 괄호로 묶고 이름을 붙인다 |
| `.groupdict()` | 이름과 값을 짝지은 딕셔너리 |
| `if m:` | 못 찾으면 `None` 이므로 값을 꺼내기 전에 확인한다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">4교시 (12:00–12:50) · 탐지 룰 ① ②</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **브루트포스** | 공격자가 무엇을 반복하는 공격인가 |
| **임계값** | 몇 번부터 수상하다고 볼지는 누가 정하나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 브루트포스 →
- 임계값 →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3 · 정규화한 기록에서 수상한 것을 찾는다</mark>


### 왜 필요한가

1. 3교시에는 로그 16줄을 같은 모양의 딕셔너리로 바꿨습니다. 이제 파일로 남긴 뒤 **셀 수 있습니다.**
2. 관제는 「무슨 일이 있었나」가 아니라 **「무엇이 수상한가」**를 봅니다. 그 기준을 코드로 적은 것이 **탐지 룰**입니다.
3. 오늘 두 가지를 만듭니다 — 한 계정을 여러 번 두드린 것(**브루트포스**)과, 한 곳에서 여러 계정을 건드린 것(**의심 IP**)입니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 탐지 룰 | 「이런 모양이면 수상하다」를 코드로 적은 기준 |
| 브루트포스 | 한 계정의 비밀번호를 여러 번 찍어 보는 공격 |
| 임계값 | 몇 번부터 수상하다고 볼지 정한 숫자 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 못 찾으면 `None` — 안 맞는 줄을 버리지 않는다</mark>

```python
import re

m = re.search(r"\d+", "숫자가 없는 문장")
print(m)             # None
```

- 못 찾으면 `None` 이 옵니다. **`None` 에서 `.group()` 을 꺼내려 하면 멈춥니다.**
- 그래서 결과는 언제나 `if m:` 으로 확인하고 씁니다.
- 안 맞은 줄을 그냥 버리면 **규칙이 틀린 것인지 원래 다른 종류인지** 알 수 없습니다. `unmatched_logs.txt` 에 따로 적어 둡니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-1 · 안 맞는 줄 세기</font></h3></td></tr></table>

`raw_logs.txt` 에서 패턴에 **안 맞는 줄이 몇 개인지** 세어 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `안 맞음 4건` |

**💡 힌트**

1. 문제 3-6의 뼈대를 그대로 씁니다.
2. `if m:` 의 반대는 `else:` 입니다.
3. `else` 안에서 숫자를 1 늘립니다.


In [26]:
# 여기에 코드를 입력하세요
unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            pass
        else:
            unmatched.append(line)

print("안 맞음", len(unmatched), "건")

안 맞음 4 건


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-2 · 안 맞는 줄을 파일로 남기기</font></h3></td></tr></table>

안 맞는 줄을 **`unmatched_logs.txt`** 에 한 줄씩 저장하시오. 저장한 뒤 `!cat unmatched_logs.txt` 로 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `안 맞음 4건을 남겼습니다` — `!cat` 으로 열면 `session closed` 세 줄과 `invalid user` 한 줄 |

**💡 힌트**

1. 안 맞는 줄을 먼저 리스트에 모읍니다.
2. 파일 쓰기는 `with open("unmatched_logs.txt", "w", encoding="utf-8") as f:` 로 엽니다.
3. 한 줄씩 쓸 때는 끝에 `\\n` 을 붙입니다.


In [29]:
# 여기에 코드를 입력하세요
with open("unmatched_logs.txt", "w", encoding="utf-8") as f:
    for line in unmatched:
        f.write(line)

print("안 맞음", len(unmatched), "건을 남겼습니다")


안 맞음 4 건을 남겼습니다


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-3 · normalize_logs.py 만들기</font></h3></td></tr></table>

오전의 산출물 **`normalize_logs.py`** 를 만드시오. 새 문법은 없습니다. 조각을 잇는 시간입니다.

> 아래 복사용 뼈대를 바로 아래 빈 코드 셀에 붙이고 `# 여기를 채우세요` 자리만 채웁니다. 미완성 코드를 실행해 기존 파일을 덮어쓰지 마세요.

1. 새 코드 셀 맨 첫 줄에 `%%writefile normalize_logs.py` 를 씁니다.
2. `import re` 와 `import json` 을 씁니다.
3. `PATTERN` 을 맨 위에 한 번 적습니다.
4. **줄 하나를 받아 딕셔너리로 돌려주는 함수를 만듭니다.** 이름은 **`parse_raw_logs`** 로 정합니다. 안 맞으면 `None` 을 돌려줍니다.
5. `raw_logs.txt` 를 한 줄씩 읽어 그 함수를 부르고, 결과가 있으면 `rows` 에, 없으면 `unmatched` 에 담습니다.
6. `rows` 를 `normalized_logs.json` 으로 저장합니다. 한글 그대로, 두 칸 들여쓰기입니다.
7. `unmatched` 를 `unmatched_logs.txt` 로 저장합니다.
8. 마지막에 `정규화 16건 / 안 맞음 4건` 을 출력합니다.

| | |
|---|---|
| 🎯 화면 | `정규화 16건 / 안 맞음 4건` |
| 🎯 파일 | `normalized_logs.json` · `unmatched_logs.txt` |

**💡 힌트**

1. 함수는 `def parse_raw_logs(line):` 로 시작하고, 안에서 `re.search` 를 부릅니다.
2. 찾았으면 `return m.groupdict()`, 못 찾았으면 `return None` 입니다.
3. 부르는 쪽에서는 `row = parse_raw_logs(line)` 뒤에 `if row:` 로 갈라 담습니다.
4. 함수 이름은 학원 교안이 정한 이름이라 그대로 씁니다.

```python
%%writefile normalize_logs.py
import re
import json

PATTERN = r""  # 여기를 채우세요

def parse_raw_logs(line):
    # 여기를 채우세요
    pass

# rows와 unmatched를 만들고 파일 두 개로 저장하는 코드를 이어서 작성합니다.
```


In [32]:
%%writefile normalize_logs.py
import re
import json

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

def parse_raw_logs(line):
    m = re.search(PATTERN, line)
    if m:
        return m.groupdict()
    return None

rows = []
unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        row = parse_raw_logs(line)
        if row:
            rows.append(row)
        else:
            unmatched.append(line)

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

with open("unmatched_logs.txt", "w", encoding="utf-8") as f:
    for line in unmatched:
        f.write(line)

print(f"정규화 {len(rows)}건 / 안 맞음 {len(unmatched)}건")

Writing normalize_logs.py


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-4 · 안 맞는 줄을 사유별로 보기</font></h3></td></tr></table>

안 맞는 줄 네 개를 화면에 출력하고, 각 줄이 **왜 안 맞았는지** 직접 읽어 보시오. 코드는 짧습니다. 읽는 것이 목적입니다.

| | |
|---|---|
| 🎯 확인 | `session closed` 줄에는 `login for` 가 없고, `invalid user` 줄에는 계정 자리에 빈칸이 들어 있다 |

**💡 힌트**

1. 문제 4-2의 `unmatched` 리스트를 그대로 씁니다.
2. `for line in unmatched:` 로 한 줄씩 출력합니다.
3. 출력한 뒤 패턴과 견줘 읽어 봅니다.


In [ ]:
# 여기에 코드를 입력하세요


아래 셀을 먼저 실행합니다. 3교시에서 만든 딕셔너리 변환을 다시 실행해 `normalized_logs.json` 을 준비합니다. 앞 문제를 못 끝냈어도 여기서부터 시작할 수 있습니다.


In [30]:
import re
import json

# 앞 문제를 끝내지 못했어도 여기서부터 시작할 수 있게, 이 셀이 파일을 만듭니다.
PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())
        else:
            unmatched.append(line.strip())

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"정규화 {len(rows)}건을 읽었습니다")


정규화 16건을 읽었습니다


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 룰 ① 브루트포스 — 계정별로 센다</mark>

세는 방법은 어제 쓴 그대로입니다. 딕셔너리에 이름을 키로 두고 1씩 더합니다.

```python
fruits = ["사과", "배", "사과"]
count = {}

for name in fruits:
    if name in count:
        count[name] = count[name] + 1
    else:
        count[name] = 1

print(count)       # {'사과': 2, '배': 1}
```

몇 번부터 수상하다고 볼지는 **사람이 정합니다.** 그 숫자를 맨 위에 한 번 적어 두는 것이 **임계값**입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-5 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
users = ["admin", "kim01", "admin"]
count = {}

for name in users:
    if name in count:
        count[name] = count[name] + 1
    else:
        count[name] = 1

print(count)
```

막히면 바로 위 `4.2 룰 ① 브루트포스` 설명을 다시 봅니다.


In [31]:
users = ["admin", "kim01", "admin"]
count = {}

for name in users:
    if name in count:
        count[name] = count[name] + 1
    else:
        count[name] = 1

print(count)


{'admin': 2, 'kim01': 1}


✅ `{'admin': 2, 'kim01': 1}`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-6 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 읽어 온 기록에서 첫 건을 봅니다.

```python
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

print(rows[0])
```

막히면 바로 위 `4.2 룰 ① 브루트포스` 설명을 다시 봅니다.


In [ ]:
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

print(rows[0])


✅ `{'time': '09:02:11', 'level': 'INFO', 'user': 'kim.cs', 'ip': '10.1.2.11'}`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-7 · 실패한 기록만 고르기</font></h3></td></tr></table>

`rows` 에서 등급이 **`WARN`** 인 기록이 몇 건인지 세어 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `실패 10건` |

**💡 힌트**

1. 셀 숫자를 담을 이름을 반복 **전에** 만들어 0 으로 둡니다 — `fail = 0`
2. `row["level"] == "WARN"` 으로 견줍니다.
3. 출력은 반복이 끝난 뒤 한 번만 합니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-8 · 계정별 실패 횟수 세기</font></h3></td></tr></table>

실패 기록을 **계정별로** 세어 딕셔너리로 만들고 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{'kim01': 1, 'park.js': 1, 'admin': 4, 'kim.cs': 1, 'lee.yh': 1, 'choi.mk': 1, 'jung.hw': 1}` |

**💡 힌트**

1. 빈 딕셔너리를 반복 전에 만듭니다.
2. `WARN` 인 기록만 셉니다.
3. 이미 있는 이름인지는 `if user in count:` 로 확인합니다.
4. 지난주에 쓴 `Counter` 로 풀어도 됩니다. 어느 쪽이든 결과는 같습니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-9 · 룰 ① 완성 — 3회 이상이면 경보</font></h3></td></tr></table>

실패가 **3회 이상**인 계정을 찾아 경보를 출력하시오. 임계값 `3` 은 맨 위에 한 번만 적습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[룰 1] 확인 필요: admin — 실패 4회` |

**💡 힌트**

1. 문제 4-8의 결과를 그대로 씁니다.
2. `for user in count:` 로 계정 이름이 하나씩 나옵니다.
3. 임계값은 `THRESHOLD = 3` 처럼 맨 위에 적습니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-10 · 임계값을 바꿔 보기</font></h3></td></tr></table>

임계값을 `2` 로 낮추면 경보가 몇 건이 되는지 확인하시오. 임계값 한 줄만 바꿔 두 번 돌려 봅니다.

| | |
|---|---|
| 🎯 확인 | 임계값 3이면 1건, 2면 그대로 1건, 1이면 7건 |

**💡 힌트**

1. 문제 4-9의 코드에서 `THRESHOLD` 숫자만 바꿉니다.
2. 경보 건수를 세는 숫자를 하나 두면 견주기 쉽습니다.
3. 임계값이 낮을수록 경보가 늘고, 그만큼 헛경보도 늡니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.3 룰 ② 의심 IP — 시선을 IP 로 뒤집는다</mark>

룰 ①은 **계정**을 기준으로 셌습니다. 같은 기록을 **IP** 기준으로 다시 보면 다른 것이 보입니다.

```python
pairs = [("A", "kim"), ("A", "lee"), ("B", "kim")]
table = {}

for key, name in pairs:
    if key in table:
        if name not in table[key]:
            table[key].append(name)
    else:
        table[key] = [name]

print(table)     # {'A': ['kim', 'lee'], 'B': ['kim']}
```

값 자리에 **리스트**를 둡니다. 같은 이름이 또 오면 넣지 않습니다.

기준을 **2개**로 잡은 것도 사람이 정한 예시입니다. 실제 환경에서는 정상 사용 패턴을 확인해 임계값을 조정합니다. 룰 ①의 임계값과 같은 이야기입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-11 · IP 마다 계정 모으기</font></h3></td></tr></table>

`rows` 를 돌면서 **IP 마다 어떤 계정이 있었는지** 리스트로 모아 출력하시오. 같은 계정은 한 번만 담습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `185.220.101.34` 에 계정 네 개가 담긴 딕셔너리 |

**💡 힌트**

1. 빈 딕셔너리를 반복 전에 만듭니다.
2. 이미 있는 IP 인지 `if ip in table:` 로 확인합니다.
3. 이미 담긴 계정인지 `if user not in table[ip]:` 로 한 번 더 확인합니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-12 · 룰 ② 완성 — 계정 2개 이상이면 경보</font></h3></td></tr></table>

한 IP 에서 **계정 2개 이상**이 나왔으면 경보를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[룰 2] 의심 IP: 185.220.101.34 — 계정 4개 시도` |

**💡 힌트**

1. 문제 4-11의 결과를 그대로 씁니다.
2. 리스트의 개수는 `len()` 으로 셉니다.
3. `for ip in table:` 로 IP 가 하나씩 나옵니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-13 · 드라이브에 남았는지 확인하기</font></h3></td></tr></table>

맨 앞에서 `agent_core` 로 들어와 있으므로, 오늘 만든 파일은 이미 드라이브에 저장돼 있습니다. 세 개가 다 있는지 확인하시오.

1. `!ls` 로 폴더 안을 봅니다.
2. `!python normalize_logs.py` 를 실행해 다시 돌아가는지 봅니다.
3. 드라이브 웹에서 `agent_core` 폴더를 열어 눈으로도 확인합니다.

| | |
|---|---|
| 🎯 확인 | `normalize_logs.py` · `normalized_logs.json` · `unmatched_logs.txt` 세 개가 보인다 |

**💡 힌트**

1. 맨 앞 준비 셀의 `%cd` 가 실행돼 있어야 합니다.
2. 출력에 `/content/drive/MyDrive/agent_core` 가 보이면 자리가 맞습니다.


In [ ]:
!ls
!python normalize_logs.py


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-14 · 두 룰을 한 번에 돌리기</font></h3></td></tr></table>

룰 ①과 룰 ②를 이어 붙여 **경보 두 줄**을 한 번에 출력하시오. 오후에 룰 ③을 여기에 더합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[룰 1] …` 과 `[룰 2] …` 두 줄 |

**💡 힌트**

1. 문제 4-9와 4-12의 코드를 순서대로 이어 붙이면 됩니다.
2. 파일을 읽는 줄은 맨 위에 한 번만 두면 됩니다.
3. 같은 이름을 두 번 쓰지 않게 `count` 와 `table` 로 나눠 둡니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `if 이름 in 딕셔너리:` | 이미 담겨 있는지 확인한다 |
| `딕셔너리[키] = 값 + 1` | 세기 |
| `딕셔너리[키] = [값]` | 값 자리에 리스트를 둔다 |
| `THRESHOLD` | 몇 번부터 수상한지는 사람이 정한다 |

오전 산출물은 드라이브 `agent_core` 폴더의 **`normalize_logs.py`** · **`normalized_logs.json`** · **`unmatched_logs.txt`** 입니다. 오후에 룰 ③을 더합니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

각 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다.


In [ ]:
#@title 정답 2-3 { display-mode: "form" }
line1 = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
line2 = "2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34"

print(len(line1.split()))
print(len(line2.split()))


In [ ]:
#@title 정답 2-4 { display-mode: "form" }
line1 = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"
line2 = "2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34"

parts1 = line1.split()
parts2 = line2.split()

print(parts1[len(parts1) - 1])
print(parts2[len(parts2) - 1])


In [ ]:
#@title 정답 2-5 { display-mode: "form" }
with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        parts = line.split()
        print(parts[6])


In [ ]:
#@title 정답 ⭐2-6 { display-mode: "form" }
count = 0

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        parts = line.split()
        if "." not in parts[6]:
            count = count + 1

print(count)


In [ ]:
#@title 정답 ⭐2-7 { display-mode: "form" }
count = 0

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        if "WARN" in line:
            count = count + 1

print(f"WARN {count}줄")


In [ ]:
#@title 정답 3-1 { display-mode: "form" }
import re

line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"

m = re.search(r"from [\d.]+", line)

print(m.group())


In [ ]:
#@title 정답 3-2 { display-mode: "form" }
import re

line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"

m = re.search(r"for [\w.]+", line)

print(m.group())


In [ ]:
#@title 정답 3-3 { display-mode: "form" }
import re

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        found = re.findall(r"from [\d.]+", line)
        print(found)


In [ ]:
#@title 정답 3-5 { display-mode: "form" }
import re

line = "2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5"

m = re.search(r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)", line)

print(m.groupdict())


In [ ]:
#@title 정답 3-6 { display-mode: "form" }
import re

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())

print(f"정규화 {len(rows)}건")


In [ ]:
#@title 정답 ⭐3-7 { display-mode: "form" }
import re

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())

users = []
for row in rows:
    users.append(row["user"])

print(users)


In [ ]:
#@title 정답 4-1 { display-mode: "form" }
import re

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

bad = 0

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            pass
        else:
            bad = bad + 1

print(f"안 맞음 {bad}건")


In [ ]:
#@title 정답 4-2 { display-mode: "form" }
import re

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            pass
        else:
            unmatched.append(line.strip())

with open("unmatched_logs.txt", "w", encoding="utf-8") as f:
    for line in unmatched:
        f.write(line + "\n")

print(f"안 맞음 {len(unmatched)}건을 남겼습니다")


In [ ]:
#@title 정답 4-3 { display-mode: "form" }
code = """import re
import json

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"


def parse_raw_logs(line):
    m = re.search(PATTERN, line)
    if m:
        return m.groupdict()
    else:
        return None


rows = []
unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        row = parse_raw_logs(line)
        if row:
            rows.append(row)
        else:
            unmatched.append(line.strip())

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

with open("unmatched_logs.txt", "w", encoding="utf-8") as f:
    for line in unmatched:
        f.write(line + chr(10))

print(f"정규화 {len(rows)}건 / 안 맞음 {len(unmatched)}건")
"""

with open("normalize_logs.py", "w", encoding="utf-8") as f:
    f.write(code)

print("normalize_logs.py 를 만들었습니다. 새 셀에서 !python normalize_logs.py 를 실행하세요.")


In [ ]:
#@title 정답 ⭐4-4 { display-mode: "form" }
import re

PATTERN = r"(?P<time>[\d:]+) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

unmatched = []
with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        if re.search(PATTERN, line):
            pass
        else:
            unmatched.append(line.strip())

for line in unmatched:
    print(line)


In [ ]:
#@title 정답 4-7 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

fail_count = 0

for row in rows:
    if row["level"] == "WARN":
        fail_count = fail_count + 1

print(f"실패 {fail_count}건")


In [ ]:
#@title 정답 4-8 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

count = {}

for row in rows:
    if row["level"] == "WARN":
        user = row["user"]
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1

print(count)


In [ ]:
#@title 정답 4-9 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3

count = {}
for row in rows:
    if row["level"] == "WARN":
        user = row["user"]
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1

for user in count:
    if count[user] >= THRESHOLD:
        print(f"[룰 1] 확인 필요: {user} — 실패 {count[user]}회")


In [ ]:
#@title 정답 ⭐4-10 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

count = {}
for row in rows:
    if row["level"] == "WARN":
        user = row["user"]
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1

for threshold in [3, 2, 1]:
    hit = 0
    for user in count:
        if count[user] >= threshold:
            hit = hit + 1
    print(f"임계값 {threshold} — 경보 {hit}건")


In [ ]:
#@title 정답 4-11 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

table = {}

for row in rows:
    ip = row["ip"]
    user = row["user"]
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

print(table)


In [ ]:
#@title 정답 4-12 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

table = {}
for row in rows:
    ip = row["ip"]
    user = row["user"]
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

for ip in table:
    if len(table[ip]) >= 2:
        print(f"[룰 2] 의심 IP: {ip} — 계정 {len(table[ip])}개 시도")


In [ ]:
#@title 정답 4-13 { display-mode: "form" }
print("맨 앞 준비 셀을 실행했다면 이미 드라이브에 저장돼 있습니다.")
print()
print("확인")
print("   !ls")
print("   !python normalize_logs.py")
print()
print("세 개가 안 보이면")
print("   맨 앞 준비 셀(%cd /content/drive/MyDrive/agent_core)을 실행하지 않은 것입니다.")
print("   그 셀을 실행한 뒤 raw_logs.txt 셀과 %%writefile 셀을 다시 실행합니다.")


In [ ]:
#@title 정답 ⭐4-14 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3

count = {}
table = {}

for row in rows:
    user = row["user"]
    ip = row["ip"]
    if row["level"] == "WARN":
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

for user in count:
    if count[user] >= THRESHOLD:
        print(f"[룰 1] 확인 필요: {user} — 실패 {count[user]}회")

for ip in table:
    if len(table[ip]) >= 2:
        print(f"[룰 2] 의심 IP: {ip} — 계정 {len(table[ip])}개 시도")
